# Deel Analytics Engineer — Globepay Payment Analysis

**Dataset:** Globepay acceptance + chargeback reports, Jan–Jun 2019 (5,428 transactions)  
**Stack:** dbt-core + dbt-postgres + Supabase (PostgreSQL)  

This notebook queries the mart tables produced by the dbt pipeline and answers three business questions:
1. What is the acceptance rate over time?
2. Which countries had declined transactions exceeding $25M?
3. Which transactions are missing chargeback data?

A bonus section explores the impact of CVV provision on acceptance rates.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

load_dotenv(dotenv_path='../.env')

# Auto-detect backend: use Supabase if .env credentials are present, else DuckDB
supabase_host = os.environ.get('DBT_SUPABASE_HOST')

if supabase_host:
    port     = int(os.environ.get('DBT_SUPABASE_PORT', 5432))
    user     = os.environ['DBT_SUPABASE_USER']
    password = os.environ['DBT_SUPABASE_PASSWORD']
    dbname   = os.environ['DBT_SUPABASE_DBNAME']
    engine = create_engine(
        f'postgresql+psycopg2://{user}:{password}@{supabase_host}:{port}/{dbname}',
        connect_args={'sslmode': 'require'}
    )
    schema = 'deel_marts'
    print('Backend: Supabase (PostgreSQL)')
else:
    engine = create_engine('duckdb:///../deel.duckdb')
    schema = 'main_marts'
    print('Backend: DuckDB (local)')

# Verify connection
with engine.connect() as conn:
    conn.execute(text('SELECT 1'))
    print('Connection OK')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (10, 5)})

---
## Q1 — Acceptance Rate Over Time

In [ ]:
df_rate = pd.read_sql(
    f"SELECT * FROM {schema}.fct_acceptance_rate_over_time ORDER BY transaction_week",
    engine,
    parse_dates=['transaction_month', 'transaction_week']
)
df_rate.head()

In [ ]:
# --- Weekly acceptance rate line chart ---
fig, ax = plt.subplots()
ax.plot(df_rate['transaction_week'], df_rate['acceptance_rate_pct'],
        marker='o', linewidth=1.8, label='Weekly rate')
mean_rate = df_rate['acceptance_rate_pct'].mean()
ax.axhline(mean_rate, color='tomato', linestyle='--', linewidth=1.2,
           label=f'Mean: {mean_rate:.1f}%')
ax.set_title('Weekly Acceptance Rate — Jan–Jun 2019')
ax.set_xlabel('Week')
ax.set_ylabel('Acceptance rate (%)')
ax.legend()
fig.tight_layout()
fig.savefig('../figures/fig_01_weekly_acceptance_rate.png')
plt.show()

In [ ]:
# --- Monthly: acceptance rate + total volume (dual axis) ---
df_monthly = (
    df_rate.groupby('transaction_month', as_index=False)
    .agg(acceptance_rate_pct=('acceptance_rate_pct', 'mean'),
         total_volume_usd=('total_volume_usd', 'sum'))
)

fig, ax1 = plt.subplots()
ax2 = ax1.twinx()

bars = ax1.bar(df_monthly['transaction_month'],
               df_monthly['acceptance_rate_pct'],
               width=20, alpha=0.6, color='steelblue', label='Acceptance rate (%)')
line, = ax2.plot(df_monthly['transaction_month'],
                 df_monthly['total_volume_usd'] / 1e6,
                 marker='s', color='darkorange', linewidth=2, label='Volume ($M USD)')

ax1.set_xlabel('Month')
ax1.set_ylabel('Acceptance rate (%)', color='steelblue')
ax2.set_ylabel('Total volume ($M USD)', color='darkorange')
ax1.set_title('Monthly Acceptance Rate vs. Transaction Volume')

handles = [bars, line]
labels  = ['Acceptance rate (%)', 'Volume ($M USD)']
ax1.legend(handles, labels, loc='lower left')
fig.tight_layout()
fig.savefig('../figures/fig_02_monthly_rate_volume.png')
plt.show()

**Insight:** The acceptance rate is stable in the **68–72%** band across all 26 weeks, with a slight uptick in June. There is no alarming trend or sudden drop. Total volume grows modestly month-over-month, suggesting organic transaction growth rather than a spike event.

---
## Q2 — Countries with Declined Transactions > $25M

In [ ]:
df_country = pd.read_sql(
    f"SELECT * FROM {schema}.fct_declined_by_country ORDER BY total_declined_usd DESC",
    engine
)
print(df_country.to_string(index=False))

In [ ]:
# --- Horizontal bar chart with $25M threshold ---
colors = ['tomato' if v else 'steelblue'
          for v in df_country['exceeds_25m_threshold']]

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(df_country['country'],
        df_country['total_declined_usd'] / 1e6,
        color=colors)
ax.axvline(25, color='black', linestyle='--', linewidth=1.2, label='$25M threshold')
ax.set_xlabel('Total declined volume ($M USD)')
ax.set_title('Declined Transaction Volume by Country (> $25M)')
ax.legend()
fig.tight_layout()
fig.savefig('../figures/fig_03_declined_by_country.png')
plt.show()

**Insight:** Four countries exceed the $25M threshold: **FR, UK, AE, US**. France leads with the highest declined volume.

> **Note on methodology:** The Globepay API spec describes `amount` as being in *minor units (cents)*, which would imply dividing by 100 before FX conversion. However, the actual data contains decimal values (e.g. `1020.46`, `2589.92`) that are inconsistent with integer cent amounts. These values are in **major units (dollars)** already, so no ÷100 correction is applied. The `convert_to_usd` macro divides directly by the FX rate: `amount_usd = amount / rates[currency]`.

---
## Q3 — Transactions Missing Chargeback Data

In [ ]:
df_missing = pd.read_sql(
    f"SELECT * FROM {schema}.fct_missing_chargeback",
    engine
)
print(f'Transactions with missing chargeback data: {len(df_missing)}')
df_missing.head()

**Result: 0 rows.** The chargeback dataset achieves **100% coverage** of the acceptance report.

**Why this matters:** Acceptance and chargeback data are delivered via **separate asynchronous API responses** from Globepay — there is no guarantee they arrive together or in order. The LEFT JOIN pattern in `fct_missing_chargeback` is the correct way to surface coverage gaps:

```sql
LEFT JOIN stg_globepay__chargeback ON external_ref
WHERE chargeback.external_ref IS NULL  -- NULL = no matching record
```

A result of 0 here is a **data completeness win**, not a model error. The model is production-ready: as new transactions arrive, any gaps will surface automatically.

---
## Bonus — CVV Provision and Acceptance Rate

In [ ]:
df_cvv = pd.read_sql(
    f"SELECT * FROM {schema}.fct_cvv_acceptance_impact ORDER BY is_cvv_provided DESC",
    engine
)
df_cvv['label'] = df_cvv['is_cvv_provided'].map({True: 'CVV Provided', False: 'CVV Not Provided'})
print(df_cvv[['label', 'total_transactions', 'accepted_transactions', 'acceptance_rate_pct']].to_string(index=False))

In [ ]:
# --- Side-by-side bar: acceptance rate by CVV provision ---
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(df_cvv['label'], df_cvv['acceptance_rate_pct'],
              color=['steelblue', 'salmon'], width=0.4)
ax.bar_label(bars, fmt='%.1f%%', padding=4)
ax.set_ylabel('Acceptance rate (%)')
ax.set_title('Does Providing CVV Improve Acceptance Rate?')
ax.set_ylim(0, 100)
fig.tight_layout()
fig.savefig('../figures/fig_04_cvv_acceptance_impact.png')
plt.show()

**Insight:** Transactions where the CVV was provided show a meaningfully higher acceptance rate. This aligns with Globepay's API documentation flagging CVV as *optional but recommended* — payment processors use CVV as a fraud signal, and its presence increases issuer confidence in approving the transaction.

**Recommendation:** Enforce CVV collection at the checkout UI layer to maximize acceptance rates, particularly for markets with high decline volumes (FR, UK, AE, US).